# Notebook 5: Regression Validation 

In order to further validate the watchlist, a regression instead of a classification. The pipeline stays the same with oof probabilities, but XGBoost will classify as regression instead of classification. 

In [1]:
#load libraries
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.model_selection import GroupKFold
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from Dataprep import preprocess_fit_transform


In [2]:
#paths (same relative layout as NB3) + parameters

FEATURES_PATH = "../../data/final_datasets/dataset_0.csv"
TARGET_PATH   = "../../data/Module 2/target_milieuschutz.csv"
CLF_OOF_PATH  = "../../models/M_oof_xgboost.csv"      

N_WATCHLIST = 25
N_SPLITS    = 5          

In [3]:
#load features + target, zero-pad plr_id, merge continuous milieu_anteil
feat = pd.read_csv(FEATURES_PATH, dtype={"plr_id": str})
tgt  = pd.read_csv(TARGET_PATH,   dtype={"plr_id": str})
feat["plr_id"] = feat["plr_id"].astype(str).str.zfill(8)
tgt["plr_id"]  = tgt["plr_id"].astype(str).str.zfill(8)

data = feat.merge(tgt[["plr_id", "milieu_anteil"]], on="plr_id", how="left")
assert data["milieu_anteil"].notna().all(), "some PLR has no milieu_anteil after merge"

In [4]:
#build X / y / groups (mirrors NB1/NB2 column definitions)
ID_COLS      = ["plr_id", "plr_name", "bez"]
GROUP_COL    = "bez"
LABEL        = "milieu_majoritaet"     # binary label; not in dataset_0, kept for parity with NB1/NB2
LEAKAGE_COLS = ["milieu_anteil"]       # continuous share the label derives from — never a feature

feature_cols = [
    c for c in data.columns
    if c not in ID_COLS + [LABEL] + LEAKAGE_COLS
]
assert not any("milieu" in c for c in feature_cols), \
    "column derived from target is in feature-set — Leakage!"
assert GROUP_COL not in feature_cols, "bez must be group only, not a feature"

X = data[feature_cols].copy()
y = data["milieu_anteil"].to_numpy()     # continuous target
groups = data[GROUP_COL].to_numpy()

print(f"{len(data)} PLR | {len(feature_cols)} features | "
      f"target range {y.min():.3f}-{y.max():.3f} | "
      f"share at 0: {(y == 0).mean():.2%}")

527 PLR | 42 features | target range 0.000-1.000 | share at 0: 59.01%


In [5]:
#OOF regression loop (unrolled, GroupKFold by Bezirk)
#Preprocessing via preprocess_fit_transform, identical args to NB2: KNN impute only, no log, no scale.
#Regressor mirrors NB2's minimal classifier (defaults + housekeeping); only the objective is swapped.
oof_pred = np.full(len(data), np.nan)
gkf = GroupKFold(n_splits=N_SPLITS)

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), start=1):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train = y[train_idx]

    #fold-wise preprocessing (leakage-safe): fit imputer on train, apply to both
    X_train_prep, X_val_prep = preprocess_fit_transform(
        X_train, X_val, feature_cols, log_cols=[], scale=False
    )

    model = XGBRegressor(
        objective="reg:logistic",   # target in [0,1]; regression analogue of the classifier
        random_state=42,
        n_jobs=-1,
    )
    model.fit(X_train_prep, y_train)
    oof_pred[val_idx] = model.predict(X_val_prep)

    #fold diagnosis: rank agreement on the held-out fold + which districts
    fold_rho, _ = spearmanr(oof_pred[val_idx], y[val_idx])
    print(f"Fold {fold}: n_val={len(val_idx):3d} | "
          f"Spearman={fold_rho:.3f} | districts={np.unique(groups[val_idx])}")

data["oof_pred"] = oof_pred
data["residual"] = data["oof_pred"] - data["milieu_anteil"]
assert data["oof_pred"].notna().all(), "some PLR never got an OOF prediction"

Fold 1: n_val= 96 | Spearman=0.772 | districts=['03 - Pankow' '10 - Marzahn-Hellersdorf']
Fold 2: n_val= 92 | Spearman=0.519 | districts=['04 - Charlottenburg-Wilmersdorf' '11 - Lichtenberg']
Fold 3: n_val=125 | Spearman=0.749 | districts=['01 - Mitte' '09 - Treptow-Köpenick' '12 - Reinickendorf']
Fold 4: n_val= 91 | Spearman=0.726 | districts=['05 - Spandau' '07 - Tempelhof-Schöneberg']
Fold 5: n_val=123 | Spearman=0.776 | districts=['02 - Friedrichshain-Kreuzberg' '06 - Steglitz-Zehlendorf'
 '08 - Neukölln']


In [6]:
#regression quality (rank-based, since this is a filter not an estimator)
rho, _ = spearmanr(data["oof_pred"], data["milieu_anteil"])
mae = np.abs(data["residual"]).mean()
print(f"pooled Spearman(oof_pred, milieu_anteil) = {rho:.3f} | MAE = {mae:.3f}")

pooled Spearman(oof_pred, milieu_anteil) = 0.705 | MAE = 0.138


In [7]:
#candidate set = undesignated only (same gate as classification: anteil <= 0.5)
#rank by positive residual = "more designation-like than its protection"
cand = data[data["milieu_anteil"] <= 0.50].copy()
cand["rank_resid"] = cand["residual"].rank(ascending=False, method="min").astype(int)
cand["rank_pred"]  = cand["oof_pred"].rank(ascending=False, method="min").astype(int)

resid_top = set(cand.nsmallest(N_WATCHLIST, "rank_resid")["plr_id"])
pred_top  = set(cand.nsmallest(N_WATCHLIST, "rank_pred")["plr_id"])

In [8]:
#compare against the classification RESEMBLANCE watchlist (oof_prob arm only),
#reconstructed from the classification OOF so it doesn't depend on how M2_watchlist
#was assembled. Undesignated PLR, top-25 by oof_prob = NB3's oof_prob arm.
#All three sets are now single top-25s -> overlaps are symmetric (X/25 both ways).
clf_oof = pd.read_csv(CLF_OOF_PATH, dtype={"plr_id": str})
clf_oof["plr_id"] = clf_oof["plr_id"].astype(str).str.zfill(8)

clf_resemblance = set(
    clf_oof[clf_oof["y_true"] == 0]
    .nlargest(N_WATCHLIST, "oof_prob")["plr_id"]
)
assert len(clf_resemblance) == N_WATCHLIST, "classification resemblance set is not 25"

#how correlated are the two model outputs at the PLR level?
merged = data.merge(clf_oof[["plr_id", "oof_prob"]], on="plr_id", how="inner")
rho_models, _ = spearmanr(merged["oof_pred"], merged["oof_prob"])


def overlap(a, b, label_a, label_b):
    inter = a & b
    print(f"{label_a}  vs  {label_b}: {len(inter)}/{len(a)} overlap "
          f"(only {label_a}: {sorted(a - b)}) "
          f"(only {label_b}: {sorted(b - a)})")


print(f"Spearman(regression oof_pred, classification oof_prob) = {rho_models:.3f}\n")
overlap(pred_top,  clf_resemblance, "reg oof_pred top-25", "clf oof_prob top-25")
overlap(resid_top, clf_resemblance, "reg residual top-25", "clf oof_prob top-25")
overlap(resid_top, pred_top,        "reg residual top-25", "reg oof_pred top-25")

Spearman(regression oof_pred, classification oof_prob) = 0.905

reg oof_pred top-25  vs  clf oof_prob top-25: 20/25 overlap (only reg oof_pred top-25: ['01100415', '03601452', '07200413', '09100305', '12100103']) (only clf oof_prob top-25: ['01200628', '04300416', '04501147', '04501150', '12100102'])
reg residual top-25  vs  clf oof_prob top-25: 17/25 overlap (only reg residual top-25: ['01100206', '01100415', '01200517', '01200629', '05300841', '09100305', '09200716', '12100103']) (only clf oof_prob top-25: ['01200628', '03601142', '04501150', '07100206', '07200414', '08100102', '09200511', '12100102'])
reg residual top-25  vs  reg oof_pred top-25: 18/25 overlap (only reg residual top-25: ['01100206', '01200517', '01200629', '04300416', '04501147', '05300841', '09200716']) (only reg oof_pred top-25: ['03601142', '03601452', '07100206', '07200413', '07200414', '08100102', '09200511'])


In [9]:
data.loc[data.plr_id.isin(['12100102','12100103']), ['plr_id','plr_name']]

,plr_id,plr_name
494,12100102,Hausotterplatz
495,12100103,Breitkopfbecken


In [11]:
#Diagnostic: are the swap PLR near-cutoff ties or real disagreements?
#Show oof_pred (reg) + oof_prob (clf) side by side with both ranks, for the
#union of the two top-25s, so the rank-25 boundary is visible.

#reconstruct classification resemblance ranking with ranks (undesignated only)
clf_res = (
    clf_oof[clf_oof["y_true"] == 0][["plr_id", "oof_prob"]]
    .copy()
)
clf_res["rank_prob"] = clf_res["oof_prob"].rank(ascending=False, method="min").astype(int)

#regression raw-resemblance ranking with ranks (same undesignated gate)
reg_res = cand[["plr_id", "plr_name", "bez", "oof_pred", "milieu_anteil"]].copy()
reg_res["rank_pred"] = reg_res["oof_pred"].rank(ascending=False, method="min").astype(int)

#join the two views
cmp = reg_res.merge(clf_res, on="plr_id", how="inner")

#the PLR that differ between reg oof_pred top-25 and clf oof_prob top-25
swap = sorted(pred_top ^ clf_resemblance)          # symmetric difference
view = (cmp[cmp["plr_id"].isin(swap)]
        .sort_values("rank_pred")
        [["plr_id", "plr_name", "bez", "oof_pred", "rank_pred", "oof_prob", "rank_prob", "milieu_anteil"]])

print(f"{len(swap)} swap PLR (in exactly one of the two raw-resemblance top-25s):\n")
print(view.to_string(index=False))

#how tight is the rank-25 boundary in each model?
p_in  = cand.nlargest(25, "oof_pred")["oof_pred"].iloc[-1]
p_out = cand.nlargest(26, "oof_pred")["oof_pred"].iloc[-1]
q_in  = clf_res.nsmallest(25, "rank_prob")["oof_prob"].iloc[-1]
q_out = clf_res.nsmallest(26, "rank_prob")["oof_prob"].iloc[-1]
print(f"\nreg oof_pred cutoff: #25={p_in:.3f} | #26={p_out:.3f} | gap={p_in-p_out:.4f}")
print(f"clf oof_prob cutoff: #25={q_in:.3f} | #26={q_out:.3f} | gap={q_in-q_out:.4f}")

10 swap PLR (in exactly one of the two raw-resemblance top-25s):

  plr_id           plr_name                             bez  oof_pred  rank_pred  oof_prob  rank_prob  milieu_anteil
03601452           Velodrom                     03 - Pankow  0.690303         10  0.008326        136       0.361355
07200413    Cheruskerstraße       07 - Tempelhof-Schöneberg  0.515034         18  0.100890         69       0.295770
12100103    Breitkopfbecken              12 - Reinickendorf  0.482257         19  0.712730         29       0.000000
09100305 Baumschulenweg Ost           09 - Treptow-Köpenick  0.446441         24  0.447517         46       0.000000
01100415    Invalidenstraße                      01 - Mitte  0.436678         25  0.693884         32       0.000000
04300416  Amtsgerichtsplatz 04 - Charlottenburg-Wilmersdorf  0.415842         28  0.862824         22       0.018465
04501147  Leon-Jessel-Platz 04 - Charlottenburg-Wilmersdorf  0.408259         29  0.984600          7       0.00000

# the regression is partially doing the protection discount inside the model, because a higher milieu_anteil is literally what it's trained to predict.

So the regression, unlike the classification, does not independently flag the PLR that just got designated as strongly designation-like. That's a point against the regression as an early-warning model, not for it — the classification model's scale_pos_weight is exactly what let it rank Leon-Jessel 7th. 

the two models agree on 20/25, but they disagree systematically on exactly the PLR the project cares most about — undesignated, zero-protection, high-resemblance — with the reweighted classifier ranking them far higher.

The Velodrom/Cheruskerstraße outliers (regression-high, non-zero share) are the other half of the same story and worth a sentence — the regression rewards existing protection, the classifier rewards resemblance regardless.

These aren't PLR straddling rank 25 by a place or two — the classification model ranks them near the very top while the regression pushes them well down. The reg cutoff gap being tiny (0.0088) is a red herring here: the swapped PLR aren't sitting at the boundary, they're far apart between the two models. So drop the "cutoff artifact" framing I suggested — the data doesn't support it.

PLR the regression ranks high but classification doesn't: Velodrom (oof_pred rank 10, oof_prob rank 136 — huge), Cheruskerstraße (18 vs 69). Both have non-zero milieu_anteil (0.36, 0.30).

PLR the classification ranks high but regression doesn't: Leon-Jessel, Lüneburger, Hausotterplatz, Wilhelmsaue, Amtsgerichtsplatz, Invalidenstraße, Breitkopfbecken — almost all milieu_anteil exactly 0.

the watchlist is stable to the target choice overall, though the binary and continuous models disagree at the margin on which of the adjacent Reinickendorf PLR — Breitkopfbecken vs Hausotterplatz — makes the raw-resemblance top-25. Both are genuine Tier 1 either way, so the substance survives.

These two are neighbouring Reinickendorf PLR that probably sit close together in feature space, so a small change in the objective (binary majority vs continuous share) is enough to flip their order right around the rank-25 cutoff. 

# raw values regression vs classification 20/25
switching the label from a 0.5 majority cut to the continuous share reshuffles only 5 of the top 25. This is the headline for the writeup.

# both reg residuals vs classification 17/25 and regression raw values vs residuals 18/25 
land on 7-8 different cases, like the urgency flag in NB3. 

--> real question is: it's "resemblance or protection-discounted," and that single choice governs ~7 PLR regardless of everything else.

Line 1 — residual top-25 vs classification union: 20/25

Take your 25 residual PLR, count how many also appear in the 32-PLR classification watchlist → 20. So 80% of the regression's residual watchlist is corroborated by the classification pipeline. The two leftover lists:

only regression (5): in your residual top-25 but not in the classification watchlist — Unter den Linden, Huttenkiez, Hansaviertel, Siemensdamm, Baumschulenweg Ost. These are the low-oof_pred(~0.37–0.40), zero-protection tail we discussed.
only classification (12): in the 32-PLR watchlist but absent from your residual top-25. That's 32 − 20 = 12. Important: this is inflated by the size gap — the classification set has 32 slots to your 25, so 7 of those 12 exist simply because it's a bigger list, not because the regression "missed" them.

# regressions regression vs  classification 
Line 2 — oof_pred top-25 vs classification union: 22/25
Same comparison, but using the regression's raw score (pred_top) instead of the residual. 22 of the 25 are in the classification watchlist — slightly higher agreement than line 1's 20.

# inside regression: residuals vs prediction 
Line 3 — residual top-25 vs oof_pred top-25: 18/25
This one is entirely inside the regression — residual ranking vs raw-score ranking, same model, same predictions. 18 PLR are in both; 25 − 18 = 7 differ on each side. This isolates the pure effect of the protection discount, with the target and model held fixed.

In [8]:
#names for the residual watchlist, for eyeballing
print(
    cand.nsmallest(N_WATCHLIST, "rank_resid")
    [["plr_id", "plr_name", "bez", "oof_pred", "milieu_anteil", "residual"]]
    .to_string(index=False)
)

  plr_id              plr_name                             bez  oof_pred  milieu_anteil  residual
01300834         Brunnenstraße                      01 - Mitte  0.749736   1.430044e-04  0.749593
01100416           Arkonaplatz                      01 - Mitte  0.689894   9.594333e-04  0.688935
01401049           Schulstraße                      01 - Mitte  0.889269   2.382715e-01  0.650997
07300619          Grazer Platz       07 - Tempelhof-Schöneberg  0.805050   1.870982e-01  0.617952
01300836 Humboldthain Nordwest                      01 - Mitte  0.711323   1.296414e-01  0.581682
01400938      Kameruner Straße                      01 - Mitte  0.803364   2.381216e-01  0.565243
04300414          Schloßstraße 04 - Charlottenburg-Wilmersdorf  0.594934   3.928747e-02  0.555647
01300731         Koloniestraße                      01 - Mitte  0.733899   2.211197e-01  0.512779
09100101           Alt-Treptow           09 - Treptow-Köpenick  0.681266   1.842958e-01  0.496970
12100103       Breit

# Claudes Results 

The regression and classification are nearly the same model. 

Spearman 0.905 at the PLR level, and the regression's raw oof_pred top-25 lands 22/25 inside your classification union. So switching from a binary majority label to the continuous share barely moves anything, the target form is close to immaterial. 

One caveat before you quote 0.905: that correlation is over all 527 PLR, which includes the ~109 designated areas that both models score high and easily agree on, so it's flattered by the easy part. The overlap numbers (20–22/25), which live in the undesignated tail where the watchlist actually operates, are the more honest measure.

The real lever is the ranking choice, not the target. Watch the decomposition: oof_pred (raw regression score) matches the classification union 22/25, but the residual matches only 20/25, and residual-vs-oof_pred within the regression is 18/25 — i.e. 7 PLR swap when you go from raw score to protection-discounted residual. That 7 is the same magnitude as the 7/25 oof_prob-vs-urgency divergence you already documented in NB3. 

It's the identical mechanism showing up independently: discounting by existing protection reshuffles ~7 of the top 25, whether you do it multiplicatively (urgency) or subtractively (residual). So this notebook doesn't just say "the model is robust" — it says the open team question (sort by oof_prob or by urgency) is exactly where the sensitivity lives, and it lives there regardless of target form or discounting formula. That's a stronger, more useful finding than a bare overlap count.

The 5 residual-only PLR are the conflation risk made concrete — and they force me to walk back my earlier suggestion. The areas the residual surfaces that the classification union never flagged (Unter den Linden, Huttenkiez, Hansaviertel, Siemensdamm, Baumschulenweg Ost) all sit at the bottom of the residual list with oof_pred ~0.37–0.40 and milieu_anteil ≈ 0. They rank on residual not because the model finds them strongly designation-like, but because there's nothing to subtract — exactly the "moderate resemblance + zero protection" case I warned about last turn and proposed a floor on oof_pred to exclude. 

But look what a floor would have done: Leon-Jessel-Platz has oof_pred 0.408, sits in precisely this bucket, and is the one that just got designated in the BVV. A floor of, say, 0.5 would have cut the single PLR that earned real-world validation. So my floor suggestion was wrong for your data — the low-oof_pred, zero-protection bucket is not just noise, and hard-thresholding it away would throw out a confirmed hit. That's worth remembering if the team debates a floor: the evidence you have argues against one.

What you can now claim, honestly: the watchlist is stable to the binary-vs-continuous target choice (Spearman 0.905, 22/25 raw overlap), the residual is a defensible weighting-free alternative to the urgency score that reproduces the same ~7-PLR protection effect, and the areas where the two disagree are low-resemblance/low-protection marginal cases — one of which just got designated. What you can't claim: that any of this validates hit rate (still the Leon-Jessel n=1 problem) or that the regression is "better" — it's equivalent, which for a robustness check is the point.